In [ ]:
###### Create Engine #####
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import text

# load environment variable from .env
load_dotenv()


db_url = os.getenv('DATABASE_URL')
engine = create_engine(db_url)


In [ ]:
print("Step 4: Building the OLAP Star Schema...")

with engine.connect() as conn:
    # 1. Create the Analytics Schema
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS olap;"))
    
    # 2. Create Dimension Tables (Using CTAS for rapid Data Warehouse population)
    # Dim_Customer
    conn.execute(text("DROP TABLE IF EXISTS olap.dim_customer CASCADE;"))
    conn.execute(text("""
        CREATE TABLE olap.dim_customer AS
        SELECT customerid, country
        FROM oltp.customers;
    """))
    conn.execute(text("ALTER TABLE olap.dim_customer ADD PRIMARY KEY (customerid);"))

    # Dim_Product
    conn.execute(text("DROP TABLE IF EXISTS olap.dim_product CASCADE;"))
    conn.execute(text("""
        CREATE TABLE olap.dim_product AS
        SELECT stockcode, description, unitprice
        FROM oltp.products;
    """))
    conn.execute(text("ALTER TABLE olap.dim_product ADD PRIMARY KEY (stockcode);"))

    # Dim_Date (A standard data warehouse practice to slice data by time periods)
    conn.execute(text("DROP TABLE IF EXISTS olap.dim_date CASCADE;"))
    conn.execute(text("""
        CREATE TABLE olap.dim_date AS
        SELECT DISTINCT 
            DATE(invoicedate) AS date_id,
            EXTRACT(YEAR FROM invoicedate) AS year,
            EXTRACT(MONTH FROM invoicedate) AS month,
            EXTRACT(QUARTER FROM invoicedate) AS quarter,
            TO_CHAR(invoicedate, 'Day') as day_of_week
        FROM oltp.orders;
    """))
    conn.execute(text("ALTER TABLE olap.dim_date ADD PRIMARY KEY (date_id);"))

    # 3. Create Fact Table (Joining our OLTP tables to pre-calculate Revenue)
    conn.execute(text("DROP TABLE IF EXISTS olap.fact_sales CASCADE;"))
    conn.execute(text("""
        CREATE TABLE olap.fact_sales AS
        SELECT 
            oi.transaction_id,
            oi.invoiceno,
            DATE(o.invoicedate) AS date_id,
            o.customerid,
            oi.stockcode,
            oi.quantity,
            (oi.quantity * p.unitprice) AS revenue
        FROM oltp.order_items oi
        JOIN oltp.orders o ON oi.invoiceno = o.invoiceno AND oi.invoicedate = o.invoicedate
        JOIN oltp.products p ON oi.stockcode = p.stockcode;
    """))
    conn.execute(text("ALTER TABLE olap.fact_sales ADD PRIMARY KEY (transaction_id);"))
    
    conn.commit()
    print("Star Schema successfully created and populated!")

In [ ]:
print("\n--- Running Management Analytics Queries ---")

# Query 1: Calculate Total Revenue and Orders by Country [cite: 69]
query_country = """
    SELECT c.country, 
           ROUND(SUM(f.revenue), 2) AS total_revenue, 
           COUNT(DISTINCT f.invoiceno) AS total_orders
    FROM olap.fact_sales f
    JOIN olap.dim_customer c ON f.customerid = c.customerid
    GROUP BY c.country
    ORDER BY total_revenue DESC
    LIMIT 5;
"""
print("\nTop 5 Countries by Revenue:")
print(pd.read_sql_query(query_country, engine))

# Query 2: Top Products by Revenue [cite: 69]
query_products = """
    SELECT p.description, 
           SUM(f.quantity) AS total_units_sold, 
           ROUND(SUM(f.revenue), 2) AS total_revenue
    FROM olap.fact_sales f
    JOIN olap.dim_product p ON f.stockcode = p.stockcode
    GROUP BY p.description
    ORDER BY total_revenue DESC
    LIMIT 5;
"""
print("\nTop 5 Products by Revenue:")
print(pd.read_sql_query(query_products, engine))

# Query 3: Monthly Sales Trend (Using our Date Dimension)
query_trends = """
    SELECT d.year, d.month, 
           ROUND(SUM(f.revenue), 2) AS monthly_revenue
    FROM olap.fact_sales f
    JOIN olap.dim_date d ON f.date_id = d.date_id
    GROUP BY d.year, d.month
    ORDER BY d.year, d.month;
"""
print("\nMonthly Revenue Trend:")
print(pd.read_sql_query(query_trends, engine))